In [ ]:
"""Build one full-band 224px cache shard on CPU (Phase 1 coverage experiment).

Identical to the cnn_topk_support_v1 cache except central_band [0,1] and slices_per_slot 20, which keeps the
sampling stride of 12 slices over the central 60%. Shard 0 also rebuilds 10 studies with the original settings and
checks them byte-for-byte against the existing 224px cache, proving the builder is unchanged."""
from pathlib import Path
import csv, hashlib, json, shutil, subprocess, sys, time
print("STARTUP: CPU-only full-band 224px cache; fixed input paths; no recursive discovery", flush=True)
def pick(paths):
    found=[p for p in paths if p.exists()]
    if len(found)!=1: raise FileNotFoundError(f"Expected one input: {paths}; found {found}")
    return found[0]
source=pick([Path('/kaggle/input/datasets/tumis001/rsna-cnn-topk-support-source-v1/skanee'),Path('/kaggle/input/rsna-cnn-topk-support-source-v1/skanee')])
data=pick([Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'),Path('/kaggle/input/rsna-knee-abnormality-detection')])
labels=pick([Path('/kaggle/input/datasets/tumis001/rsna-knee-hpo-v31-d2-report-labels/soft_labels_hpo_v3_1_d2_structured.json'),Path('/kaggle/input/rsna-knee-hpo-v31-d2-report-labels/soft_labels_hpo_v3_1_d2_structured.json')])
assert hashlib.sha256(labels.read_bytes()).hexdigest()=='38367d62eb9e7c198b69d88e2b9ad419e00fea9d1211b98feae45cd8a386f92d'
label_ids={r['study_id'] for r in json.loads(labels.read_text())['labels']};assert len(label_ids)==4349
work=Path('/kaggle/working/fullband224');work.mkdir(parents=True,exist_ok=True)
writable=Path('/kaggle/working/fullband224_source');shutil.copytree(source,writable,dirs_exist_ok=True)
builder_text = '"""Cache physically cropped uint8 slices for trainable-backbone baselines.\n\nNo model runs here, so this needs no gated checkpoint and no internet. Output\nper shard: ``slices_NNN.npy`` (uint8 [slices, size, size]) plus one\n``manifest.jsonl`` line per (study, series slot) pointing at its rows.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom src.features.dicom import decode_pixels, normalized_positions, read_series, series_bounds\nfrom src.images.prepare import (physical_crop_with_support, pixel_spacing, select_series,\n                                select_slice_indices, to_uint8)\nfrom src.reports.schema import TARGETS\n\n\ndef read_csv(path: Path) -> list[dict[str, str]]:\n    with path.open(newline="", encoding="utf-8-sig") as stream:\n        return list(csv.DictReader(stream))\n\n\nclass SlotDataset:\n    """One (study, series slot) per item, decoded in DataLoader workers."""\n\n    def __init__(self, jobs: list[dict], cache: dict) -> None:\n        self.jobs, self.cache = jobs, cache\n\n    def __len__(self) -> int:\n        return len(self.jobs)\n\n    def __getitem__(self, index: int):\n        import pydicom\n\n        job = self.jobs[index]\n        ordered = read_series(Path(job["folder"]))\n        record = {**{k: job[k] for k in ("study_id", "series_id", "slot", "plane", "fluid_sensitive")},\n                  "order_source": ordered.order_source, "exceptions": ordered.exceptions,\n                  "decode_errors": 0, "files": [], "positions": [], "spacing_mm": None}\n        if ordered.order_source in self.cache["exclude_order_sources"] or not ordered.files:\n            empty = np.zeros((0, self.cache["image_size"], self.cache["image_size"]), dtype=np.uint8)\n            return record, empty, empty.copy()\n\n        wanted = select_slice_indices(len(ordered.files), self.cache["slices_per_slot"],\n                                      *self.cache["central_band"])\n        positions = normalized_positions(ordered.coordinates)\n        images, supports, decoded, spacing = [], [], [], None\n        for position in wanted:\n            path = ordered.files[position]\n            try:\n                dicom = pydicom.dcmread(path)\n                pixels = decode_pixels(dicom)\n                if pixels.ndim != 2:\n                    raise ValueError("expected a 2D slice")\n                spacing = spacing or pixel_spacing(dicom)\n                decoded.append((position, pixels, getattr(dicom, "PhotometricInterpretation", "") == "MONOCHROME1"))\n            except Exception:\n                record["decode_errors"] += 1\n        if decoded:\n            low, high = series_bounds([pixels for _, pixels, _ in decoded], *self.cache["clip_percentiles"])\n            for position, pixels, inverted in decoded:\n                support = np.isfinite(pixels)\n                eight_bit = to_uint8(pixels, low, high, inverted)\n                # A padding marker may map to an intensity endpoint for\n                # MONOCHROME1. Give all unsupported locations one fixed value;\n                # the parallel support mask retains the distinction from a\n                # genuine zero-valued acquired pixel.\n                eight_bit[~support] = 0\n                cropped, cropped_support = physical_crop_with_support(\n                    eight_bit.astype(np.float32), support, spacing,\n                    self.cache["field_of_view_mm"], self.cache["image_size"]\n                )\n                images.append(cropped.round().astype(np.uint8))\n                supports.append(cropped_support)\n                record["files"].append(ordered.files[position].name)\n                record["positions"].append(positions[position])\n        record["spacing_mm"] = list(spacing) if spacing else None\n        stacked = np.stack(images) if images else np.zeros(\n            (0, self.cache["image_size"], self.cache["image_size"]), dtype=np.uint8)\n        support_stack = np.stack(supports) if supports else np.zeros_like(stacked)\n        return record, stacked, support_stack\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--data-root", type=Path, required=True)\n    parser.add_argument("--split", choices=("train", "test"), default="train")\n    parser.add_argument("--subset", choices=("gold", "weak", "all"), default="weak")\n    parser.add_argument("--config", type=Path, default=Path("configs/cnn_baseline_v1.json"))\n    parser.add_argument("--output-dir", type=Path, required=True)\n    parser.add_argument("--shard-index", type=int, default=0)\n    parser.add_argument("--shard-count", type=int, default=1)\n    parser.add_argument("--limit", type=int, default=None)\n    parser.add_argument("--workers", type=int, default=4)\n    parser.add_argument("--studies-per-part", type=int, default=400)\n    parser.add_argument("--time-budget-hours", type=float, default=11.0)\n    args = parser.parse_args()\n    started = time.monotonic()\n\n    import torch\n\n    config = json.loads(args.config.read_text(encoding="utf-8"))\n    cache = config["cache"]\n    out = args.output_dir\n    out.mkdir(parents=True, exist_ok=True)\n    manifest_path = out / "manifest.jsonl"\n    done = set()\n    if manifest_path.is_file():\n        with manifest_path.open(encoding="utf-8") as stream:\n            done = {json.loads(line)["study_id"] for line in stream if line.strip()}\n\n    studies = read_csv(args.data_root / f"{args.split}.csv")\n    series = read_csv(args.data_root / f"{args.split}_series.csv")\n    if args.subset != "all":\n        if args.split != "train":\n            raise ValueError("gold/weak subsets exist only for train")\n        is_gold = lambda row: all(row.get(t, "") in {"0", "1"} for t in TARGETS)\n        studies = [row for row in studies if is_gold(row) == (args.subset == "gold")]\n    wanted = sorted(row["StudyInstanceUID"] for row in studies)\n    wanted = [s for s in wanted\n              if int(hashlib.sha256(s.encode()).hexdigest(), 16) % args.shard_count == args.shard_index]\n    wanted = [s for s in wanted if s not in done][:args.limit]\n    by_study: dict[str, list[dict]] = {}\n    for row in series:\n        by_study.setdefault(row["StudyInstanceUID"], []).append(row)\n\n    jobs = []\n    for study_id in wanted:\n        for slot, row in enumerate(select_series(by_study.get(study_id, []), cache["series_slots"])):\n            jobs.append({"study_id": study_id, "series_id": row["SeriesInstanceUID"], "slot": slot,\n                         "folder": str(args.data_root / f"{args.split}_series" / study_id / row["SeriesInstanceUID"]),\n                         "plane": row.get("Anatomical_Plane"), "fluid_sensitive": row.get("Fluid_Sensitive")})\n    print(f"{len(wanted)} studies, {len(jobs)} series slots", flush=True)\n    loader = torch.utils.data.DataLoader(SlotDataset(jobs, cache), batch_size=None,\n                                         num_workers=args.workers,\n                                         prefetch_factor=4 if args.workers else None)\n\n    part_index = len(list(out.glob("slices_*.npy")))\n    buffer: list[np.ndarray] = []\n    support_buffer: list[np.ndarray] = []\n    pending: list[dict] = []\n    rows_in_part = 0\n    stats = {"slots": 0, "slices": 0, "decode_errors": 0, "skipped_slots": 0, "no_spacing": 0}\n\n    def flush() -> None:\n        nonlocal part_index, buffer, support_buffer, pending, rows_in_part\n        if not pending:\n            return\n        name = f"slices_{part_index:03d}.npy"\n        support_name = f"support_{part_index:03d}.npy"\n        stacked = (np.concatenate(buffer) if buffer\n                   else np.zeros((0, cache["image_size"], cache["image_size"]), dtype=np.uint8))\n        stacked_support = (np.concatenate(support_buffer) if support_buffer\n                           else np.zeros_like(stacked))\n        np.save(out / name, stacked)\n        np.save(out / support_name, stacked_support)\n        with manifest_path.open("a", encoding="utf-8") as stream:\n            for record in pending:\n                stream.write(json.dumps({**record, "part": name,\n                                         "support_part": support_name}) + "\\n")\n        part_index += 1\n        buffer, support_buffer, pending, rows_in_part = [], [], [], 0\n\n    current = None\n    for record, images, supports in loader:\n        if record["study_id"] != current:\n            if time.monotonic() - started > args.time_budget_hours * 3600:\n                print("Time budget reached; stopping at a study boundary", flush=True)\n                break\n            if len({r["study_id"] for r in pending}) >= args.studies_per_part:\n                flush()\n            current = record["study_id"]\n        if len(images):\n            buffer.append(images)\n            support_buffer.append(supports)\n        else:\n            stats["skipped_slots"] += 1\n        record.update(row_start=rows_in_part, count=len(images))\n        rows_in_part += len(images)\n        pending.append(record)\n        stats["slots"] += 1\n        stats["slices"] += len(images)\n        stats["decode_errors"] += record["decode_errors"]\n        stats["no_spacing"] += record["spacing_mm"] is None\n        if stats["slots"] % 50 == 0:\n            elapsed = time.monotonic() - started\n            print(f"{stats[\'slots\']}/{len(jobs)} slots, {stats[\'slices\']} slices, "\n                  f"{stats[\'slices\'] / elapsed:.1f} slices/s", flush=True)\n    flush()\n\n    summary = {"version": config["version"], "cache": cache, "split": args.split, "subset": args.subset,\n               "shard": [args.shard_index, args.shard_count], "run_stats": stats,\n               "config_sha256": hashlib.sha256(args.config.read_bytes()).hexdigest(),\n               "seconds": round(time.monotonic() - started, 1)}\n    with manifest_path.open(encoding="utf-8") as stream:\n        summary["studies_in_manifest"] = len({json.loads(line)["study_id"] for line in stream if line.strip()})\n    (out / "cache_summary.json").write_text(json.dumps(summary, indent=2) + "\\n", encoding="utf-8")\n    print(json.dumps(summary, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n'
builder_path=writable/'scripts/build_slice_cache.py';builder_path.write_text(builder_text)
config=json.loads((source/'configs/cnn_topk_support_v1.json').read_text())
assert config['cache']['central_band']==[0.2,0.8] and config['cache']['slices_per_slot']==12 and config['cache']['image_size']==224
original_config_path=work/'original_cache_config.json';original_config_path.write_text(json.dumps(config,indent=2)+'\n')
config['version']='cnn_topk_support_fullband_v1';config['cache']['central_band']=[0.0,1.0];config['cache']['slices_per_slot']=20
config_path=work/'cache_config.json';config_path.write_text(json.dumps(config,indent=2)+'\n')
SHARD_INDEX = 1
if SHARD_INDEX==0:
    import numpy as np
    reference=pick([Path('/kaggle/input/datasets/tumis001/rsna-cnn-topk-support-cache-shard-0-rehost'),Path('/kaggle/input/rsna-cnn-topk-support-cache-shard-0-rehost')])
    parity=work/'parity_original_settings'
    subprocess.run([sys.executable,'-u',str(builder_path),'--data-root',str(data),'--subset','weak','--config',str(original_config_path),'--shard-index','0','--shard-count','3','--output-dir',str(parity),'--workers','4','--limit','10'],check=True)
    ref_rows={(r['study_id'],r['slot']):r for r in map(json.loads,(reference/'manifest.jsonl').read_text().splitlines()) if r}
    checked=0
    for r in map(json.loads,(parity/'manifest.jsonl').read_text().splitlines()):
        q=ref_rows[(r['study_id'],r['slot'])]
        assert r['files']==q['files'] and r['count']==q['count'],(r['study_id'],r['slot'])
        for part,key in (('part','slices'),('support_part','support')):
            a=np.load(parity/r[part],mmap_mode='r')[r['row_start']:r['row_start']+r['count']]
            b=np.load(reference/q[part],mmap_mode='r')[q['row_start']:q['row_start']+q['count']]
            assert np.array_equal(a,b),(r['study_id'],r['slot'],key)
        checked+=1
    shutil.rmtree(parity)
    print(f'PARITY_OK: {checked} series slots rebuilt with original settings match the existing 224px cache byte-for-byte',flush=True)
    (work/'parity.json').write_text(json.dumps({'parity_slots_checked':checked,'match':True})+'\n')
started=time.monotonic();all_ids=set();reports=[]
for shard in [SHARD_INDEX]:
    remaining=10.5-(time.monotonic()-started)/3600
    if remaining<0.1: break
    out=work/f'shard_{shard}'/'slice_cache'
    print(f"START shard={shard}/3; remaining hours={remaining:.2f}",flush=True)
    subprocess.run([sys.executable,'-u',str(builder_path),'--data-root',str(data),'--subset','weak','--config',str(config_path),'--shard-index',str(shard),'--shard-count','3','--output-dir',str(out),'--workers','4','--studies-per-part','100','--time-budget-hours',str(remaining)],check=True)
    records=[json.loads(line) for line in (out/'manifest.jsonl').read_text().splitlines() if line.strip()]
    ids={r['study_id'] for r in records};expected={s for s in label_ids if int(hashlib.sha256(s.encode()).hexdigest(),16)%3==shard}
    if ids-expected or all_ids&ids: raise ValueError('Unexpected/duplicate cached IDs')
    import numpy as np
    for file in sorted(out.glob('*.npy')):
        array=np.load(file,mmap_mode='r');assert array.dtype==np.uint8 and array.shape[1:]==(224,224)
    all_ids.update(ids);reports.append({'shard':shard,'expected':len(expected),'actual':len(ids),'complete':ids==expected})
    (work/'completion.json').write_text(json.dumps({'shards':reports,'total':len(all_ids),'complete':all(r['complete'] for r in reports)},indent=2)+'\n')
    if ids!=expected:
        print('INCOMPLETE: time budget; preserve output for continuation',flush=True);break
print(json.dumps({'shards':reports,'total':len(all_ids),'complete':all(r['complete'] for r in reports)}),flush=True)

